This is an unedited copy of the research code I used for OCR'ing the French corpus. I ran this code in Google Colab with vertexai but you can run it in your computer as well. You will need to do some edits for that.

A great place to start is with examples from google itself
- https://github.com/google-gemini/cookbook
- https://ai.google.dev/gemini-api/docs
- https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started.ipynb

In [ ]:
import os
import sys

from IPython.display import HTML, Markdown, display
from google import genai
from google.genai import types
from google.genai.types import GenerateContentConfig, Part
from pydantic import BaseModel

import json
import pypdf
from pypdf import PdfReader, PdfWriter
import time
import pathlib
from pathlib import Path
import re

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

In [ ]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = "global"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

In [ ]:
# List all available Gemini models
for model in client.models.list():
    print(f"Model: {model.name}")
    print("---")

Model: publishers/google/models/gemini-1.5-pro-002
---
Model: publishers/google/models/gemini-2.0-flash-001
---
Model: publishers/google/models/gemini-2.0-flash
---
Model: publishers/google/models/gemini-2.0-flash-lite-001
---
Model: publishers/google/models/gemini-2.5-flash-preview-04-17
---
Model: publishers/google/models/gemini-2.5-pro-exp-03-25
---
Model: publishers/google/models/gemini-2.5-pro
---
Model: publishers/google/models/gemini-2.5-flash
---
Model: publishers/google/models/gemini-2.5-flash-lite
---
Model: publishers/google/models/gemini-2.5-pro-tts
---
Model: publishers/google/models/gemini-2.5-flash-tts
---
Model: publishers/google/models/gemini-live-2.5-flash-native-audio
---
Model: publishers/google/models/gemini-3-flash-preview
---
Model: publishers/google/models/gemini-3.1-flash-lite-preview
---
Model: publishers/google/models/gemini-3.1-flash-image-preview
---
Model: publishers/google/models/gemini-3.1-pro-preview
---
Model: publishers/google/models/gemini-embedding-

In [ ]:
MODEL_ID = "gemini-3-flash-preview"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

# go to specific location
path = '/content/drive/MyDrive/merve-february2026'
os.chdir(path)

# Confirm the change
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/merve-february2026


In [ ]:
# Split the PDF into pages

def extract_pdf_pages(pdf_path, output_dir, starting_page_num=1):
    """
    Extract all pages from a PDF and save them as individual PDF files.

    Parameters:
    -----------
    pdf_path : str
        Path to the input PDF file
    starting_page_num : int, default=1
        The starting page number for naming the output files

    Returns:
    --------
    str
        Path to the output folder containing the individual page PDFs

    Example:
    --------
    >>> extract_pdf_pages("document.pdf", starting_page_num=5)
    # This will create files: page_5.pdf, page_6.pdf, page_7.pdf, etc.
    """

    # Hardcoded output folder name
    output_folder = "transliteration_pages"

    output_path = output_dir + "/" + output_folder

    # Create output folder if it doesn't exist
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    # Read the PDF
    reader = PdfReader(pdf_path)
    total_pages = len(reader.pages)

    print(f"Processing {total_pages} pages from '{pdf_path}'...")

    # Extract each page
    for page_index in range(total_pages):
        # Create a new PDF writer for each page
        writer = PdfWriter()

        # Add the current page to the writer
        writer.add_page(reader.pages[page_index])

        # Calculate the output page number
        output_page_num = starting_page_num + page_index

        # Create output filename
        output_filename = f"page_{output_page_num}.pdf"
        output_page_path = os.path.join(output_path, output_filename)

        # Write the page to a new PDF file
        with open(output_page_path, 'wb') as output_file:
            writer.write(output_file)

    print(f"\nDone! All {total_pages} pages saved to '{output_path}/' folder")
    return output_folder

In [ ]:
extract_pdf_pages('where/your/pdf/is', 'where/the/folder/for/pdf/pages/should/go', 'page number to start enumeration')

In [ ]:
# Directory containing the examples
examples_dir = 'montagu_french_fewshot'

# Get all PDF files in the directory
pdf_files = [f for f in os.listdir(examples_dir) if f.endswith('.pdf')]

# Extract page numbers from filenames
page_numbers = []
for pdf_file in pdf_files:
    # Extract number from filename like "page_35.pdf"
    page_num = int(pdf_file.replace('page_', '').replace('.pdf', ''))
    page_numbers.append(page_num)

# Sort page numbers for consistent ordering
page_numbers.sort()

# Dictionary to store PDF data as Parts
example_pdfs = {}

# Load each PDF as bytes and create Parts
for page_num in page_numbers:
    pdf_path = os.path.join(examples_dir, f'page_{page_num}.pdf')

    with open(pdf_path, 'rb') as f:
        file_bytes = f.read()

    # Store as Part objects
    example_pdfs[page_num] = Part.from_bytes(
        data=file_bytes,
        mime_type="application/pdf"
    )

# Load the corresponding JSON files and combine with PDFs
example_parts = []
for page_num in page_numbers:
    json_path = os.path.join(examples_dir, f'page_{page_num}.json')

    with open(json_path, 'r', encoding='utf-8') as f:
        output_data = json.load(f)

    output_json = json.dumps(output_data, ensure_ascii=False, indent=2)

    example_parts.append({
        "file": example_pdfs[page_num],
        "page": page_num,
        "output_json": output_json
    })

print(f"Loaded {len(example_parts)} examples from {examples_dir}/")
print(f"Pages: {page_numbers}")

Loaded 6 examples from montagu_french_fewshot/
Pages: [16, 17, 22, 24, 29, 31]


In [ ]:
# for montagu

instructions = """You are a specialist in 18th and 19th-century French and English typography.
Transcribe the scanned page exactly as instructed and return ONLY valid JSON — no markdown fences, no explanation.

STRUCTURE
---------
Every page returns a JSON object with these top-level fields:

  page          : Page number exactly as printed. Integer for arabic numerals (24, 31),
                  string for roman numerals ("viij", "ix"). Do not infer — read it from the page.
  language      : The language of the letter, either English marked as "en" or French marked as "fr"
  letters       : Always a list. Never omit this field. Rules below.

LETTERS LIST
------------
Each item in "letters" represents one letter's content visible on this page.

  number    : The letter number as printed (e.g. "V", "VI", "IV"). null if this item is a
              continuation fragment with no letter opening on this page.
  heading   : The heading line as printed (e.g. "LETTRE V.", "IV. TO THE LADY RICH.*",
              "IV. A LADY RICH*."). null if number is null.
  recipient : The recipient line as printed (e.g. "A la Comtesse de B***.", "A Madame P***.").
              null if number is null. For editions where the recipient is embedded in the heading
              line rather than printed separately, set to null.
  location  : The place of writing as printed (e.g. "Nuremberg", "Cologn", "Cologne"). null if
              number is null.
  date      : The date line as printed (e.g. "le 22 Août 1716. Vieux style.",
              "Aug. 16, O. S. 1716.", "le 16 août 1716, V. S."). null if number is null.
  body      : All body text of this letter visible on this page. Never null — use "" only if
              genuinely empty. If the body ends mid-word due to a page break, include the
              hyphenated fragment (e.g. "...femme de l'En-", "...that no enchant-").
  footnote  : A single string containing the footnote(s) for this letter on this page. If there
              are multiple footnotes for one letter on one page, concatenate them separated by
              " / ". null if there is no footnote.

HOW MANY ITEMS IN LETTERS
--------------------------
- Pure continuation page (no letter opens, no letter closes): one item, all metadata null.
- Page where a single letter opens: one item, with number/heading/recipient/location/date filled.
- Page where one letter ends and another begins: two items. The first item has all metadata null
  (it is the tail of the previous letter). The second item has full metadata for the new letter.
- Section/front-matter page: one item with all metadata null, body contains the prose text.
- Blank page: empty list [].

RUNNING HEADERS
---------------
Running headers (e.g. "LETTRE IV." or "LETTER XLII." printed in the top margin) are NOT
letter headings. Ignore them entirely — do not put them in heading and do not let them
influence number or any other field.

QUIRE SIGNATURES
----------------
Short letter+numeral marks printed below and to the right of the text block (e.g. "B iij",
"B iv", "A iv", "C", "D iv") are printer's gathering marks. Ignore them entirely — they are
not part of the body text and must not appear anywhere in the output.

IMAGES
----------------
There are occasional etchings and other forms of adornment. Ignore them entirely.

ORTHOGRAPHY
-----------
- Normalize long-s (ſ) to regular s.
- Preserve & (ampersand), all accents, and period punctuation faithfully.
- Do not modernize spelling in either language.
- For the bilingual 1816 edition: transcribe each page exactly as printed in its own language
  with no normalization.

FOOTNOTES
---------
- Footnote markers (* † ‡ or numbers) must be preserved inline in body exactly where they appear.
- The full footnote text goes in the footnote field of the letter item they annotate.
- If a footnote belongs to a continuation fragment (number: null), place it in that item's
  footnote field.

NULL VS ABSENT
--------------
All fields listed above must always be present. Use null explicitly — never omit a field.
footnote is null (not "") when absent. body is never null — use "" only for genuinely empty pages."""

In [ ]:
def process_page(page_num, client, model_id, instructions, example_parts, base_dir):
    """
    Process a single page and return both extracted JSON and usage metadata.

    Args:
        page_num: Page number to process
        client: Gemini client
        model_id: Model identifier
        instructions: Processing instructions
        example_parts: Few-shot examples
        base_dir: Base directory containing the transliteration_pages folder

    Returns: (success: bool, data: dict)
    """

    pdf_path = Path(base_dir) / 'transliteration_pages' / f'page_{page_num}.pdf'

    if not pdf_path.exists():
        return False, {"page": page_num, "error": f"PDF not found: {pdf_path}"}

    # Read the PDF as bytes
    try:
        with open(pdf_path, "rb") as f:
            page_bytes = f.read()
    except Exception as e:
        return False, {"page": page_num, "error": f"Failed to read PDF: {str(e)}"}

    page_part = Part.from_bytes(data=page_bytes, mime_type="application/pdf")

    # Build the prompt using the few-shot examples
    contents = [instructions, "\n<EXAMPLES>\n"]
    for part in example_parts:
        contents.extend([
            part["file"],
            f"Output for page {part['page']}:",
            part["output_json"],
        ])
    contents.extend(["\n</EXAMPLES>\n", "\n--- PROCESS THIS PAGE ---\n"])
    contents.append(page_part)

    # Call Gemini
    try:
        response = client.models.generate_content(
            model=model_id,
            contents=contents,
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level="low"),
                safety_settings=[
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                    ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                    ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                    ),
                    types.SafetySetting(
                        category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
                        threshold=types.HarmBlockThreshold.BLOCK_NONE,
                    ),
                ],
            ),
        )

        # Extract response text safely
        try:
            raw_text = response.text.strip() if response.text else ""
        except (ValueError, AttributeError):
            raw_text = ""

        # Get finish reason
        finish_reason = (
            response.candidates[0].finish_reason.name
            if response.candidates
            else "NO_CANDIDATES"
        )

        # Determine success based on whether we got text
        is_successful = bool(raw_text)

        result = {
            "page": page_num,
            "raw_response": raw_text,
            "finish_reason": finish_reason,
            "usage": {
                "prompt_tokens": response.usage_metadata.prompt_token_count,
                "thought_tokens": response.usage_metadata.thoughts_token_count,
                "output_tokens": response.usage_metadata.candidates_token_count,
                "total_tokens": response.usage_metadata.total_token_count,
            }
        }

        # Add safety info if response was blocked/empty
        if not is_successful:
            try:
                # Try to get safety ratings from candidates first
                if hasattr(response, 'candidates') and response.candidates:
                    result["safety_ratings"] = [
                        {"category": rating.category.name, "probability": rating.probability.name}
                        for rating in response.candidates[0].safety_ratings
                    ]

                # Also check prompt_feedback for blocking info (often contains safety data when NO_CANDIDATES)
                if hasattr(response, 'prompt_feedback') and response.prompt_feedback:
                    prompt_feedback = {}

                    if hasattr(response.prompt_feedback, 'block_reason'):
                        prompt_feedback["block_reason"] = response.prompt_feedback.block_reason.name

                    if hasattr(response.prompt_feedback, 'safety_ratings'):
                        prompt_feedback["safety_ratings"] = [
                            {"category": rating.category.name, "probability": rating.probability.name}
                            for rating in response.prompt_feedback.safety_ratings
                        ]

                    if prompt_feedback:
                        result["prompt_feedback"] = prompt_feedback

            except Exception as e:
                # Don't fail the whole request if we can't get safety info
                result["safety_info_error"] = str(e)

            # Add error field
            result["error"] = f"Empty response (finish_reason: {finish_reason})"

        return is_successful, result

    except Exception as e:
        return False, {"page": page_num, "error": str(e)}


In [ ]:
def batch_process_pages(start_page, end_page, client, model_id, instructions, example_parts, base_dir, skip_existing=True):
    """
    Process pages with comprehensive retry logic.

    Retries for:
    - Rate limit errors (429/RESOURCE_EXHAUSTED)
    - Empty responses (blocked/safety issues)
    - API errors (NoneType, etc.)
    """
    output_dir = Path(base_dir) / 'gemini_output'
    output_dir.mkdir(parents=True, exist_ok=True)

    jsonl_file = output_dir / 'all_responses.jsonl'
    results_summary = []

    # Retry parameters
    base_delay = 5
    max_delay = 120
    max_retries_rate_limit = 5
    max_retries_other = 3

    for page_num in range(start_page, end_page + 1):
        individual_file = output_dir / f'page_{page_num}.json'

        if skip_existing and individual_file.exists():
            print(f"⊘ Page {page_num} already exists, skipping...")
            results_summary.append({
                "page": page_num,
                "status": "skipped",
                "reason": "already exists"
            })
            continue

        print(f"Processing page {page_num}...")

        # Retry loop
        retry_count = 0
        success = False
        final_data = None
        wait_time = base_delay

        while retry_count < max_retries_rate_limit:
            success, data = process_page(
                page_num, client, model_id, instructions, example_parts, base_dir
            )

            error_msg = data.get("error", "")

            # Determine if we should retry
            should_retry = False
            retry_type = None

            if not success:
                if "RESOURCE_EXHAUSTED" in error_msg or "429" in error_msg:
                    should_retry = True
                    retry_type = "rate_limit"
                    max_retries = max_retries_rate_limit
                    retry_wait = min(base_delay * (2 ** retry_count), max_delay)
                elif "Empty response" in error_msg or "NO_CANDIDATES" in error_msg:
                    should_retry = True
                    retry_type = "empty_response"
                    max_retries = max_retries_other
                    retry_wait = 2
                elif "not subscriptable" in error_msg or "NoneType" in error_msg:
                    should_retry = True
                    retry_type = "api_error"
                    max_retries = max_retries_other
                    retry_wait = 2
                elif "PDF not found" in error_msg:
                    # Don't retry for missing files
                    should_retry = False
                else:
                    # Unknown error - try once more
                    should_retry = retry_count == 0
                    retry_type = "unknown"
                    max_retries = 1
                    retry_wait = 2

            if should_retry and retry_count < max_retries:
                retry_count += 1
                print(f"⚠ {retry_type} error (attempt {retry_count}/{max_retries}): {error_msg}")
                print(f"Retrying in {retry_wait} seconds...")
                time.sleep(retry_wait)
                continue
            else:
                # Either success or max retries reached
                final_data = data
                break

        # Save results
        if success:
            # Save to JSONL
            with open(jsonl_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

            # Save individual file
            with open(individual_file, 'w', encoding='utf-8') as f:
                json.dump(final_data, f, ensure_ascii=False, indent=2)

            retry_msg = f" (after {retry_count} retries)" if retry_count > 0 else ""
            print(f"✓ Page {page_num} saved{retry_msg}")
            results_summary.append({
                "page": page_num,
                "status": "success",
                "retries": retry_count
            })
            wait_time = base_delay

        else:
            print(f"✗ Page {page_num} failed: {final_data.get('error', 'Unknown error')}")

            # Save failed attempt to JSONL for debugging
            with open(jsonl_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

            results_summary.append({
                "page": page_num,
                "status": "failed",
                "error": final_data.get('error'),
                "finish_reason": final_data.get('finish_reason'),
                "retries": retry_count
            })

            # Exponential backoff for consecutive failures
            wait_time = min(base_delay * 2, max_delay)

        # Rate limiting between pages
        if page_num < end_page:
            print(f"Waiting {wait_time} seconds before next page...")
            time.sleep(wait_time)

    # Save summary
    summary_file = output_dir / 'processing_summary.json'
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump({
            "total_pages": len(results_summary),
            "successful": sum(1 for r in results_summary if r["status"] == "success"),
            "failed": sum(1 for r in results_summary if r["status"] == "failed"),
            "skipped": sum(1 for r in results_summary if r["status"] == "skipped"),
            "results": results_summary
        }, f, indent=2)

    print(f"\n{'='*50}")
    print(f"Processing complete!")
    print(f"Successful: {sum(1 for r in results_summary if r['status'] == 'success')}")
    print(f"Failed: {sum(1 for r in results_summary if r['status'] == 'failed')}")
    print(f"Skipped: {sum(1 for r in results_summary if r['status'] == 'skipped')}")
    print(f"Results saved to: {output_dir}")
    print(f"{'='*50}")

    return results_summary


In [ ]:
base_directory = "where/your/split/pdf/pages/are"

In [ ]:
# Example usage:

# Process the first 5 pages
results = batch_process_pages(
    start_page=9,
    end_page=15,
    client=client,
    model_id=MODEL_ID,
    instructions=instructions,
    example_parts=example_parts,
    base_dir=base_directory
)

In [ ]:
def get_remaining_pages(base_dir):
    """
    Identify which pages have already been processed and which remain.

    Args:
        base_dir: Base directory containing transliteration_pages and gemini_output folders

    Returns:
        tuple: (processed_pages, remaining_pages, all_page_numbers)
    """
    base_path = Path(base_dir)

    # Check which pages have already been processed
    output_dir = base_path / 'gemini_output'
    if output_dir.exists():
        processed_pages = {int(p.stem.split('_')[1]) for p in output_dir.glob('page_*.json')}
    else:
        processed_pages = set()

    print(f"Already processed: {sorted(processed_pages) if processed_pages else 'None'}")

    # Get all available page PDFs
    transliteration_dir = base_path / 'transliteration_pages'
    if transliteration_dir.exists():
        page_files = os.listdir(transliteration_dir)
        all_page_numbers = [int(re.search(r'page_(\d+)\.pdf', f).group(1))
                            for f in page_files
                            if f.startswith("page_") and f.endswith(".pdf")]
    else:
        print(f"Warning: {transliteration_dir} does not exist!")
        all_page_numbers = []

    # Calculate remaining pages
    if all_page_numbers:
        remaining_pages = [p for p in range(min(all_page_numbers), max(all_page_numbers) + 1)
                           if p not in processed_pages]
        print(f"Total pages available: {len(all_page_numbers)}")
        print(f"Remaining to process: {len(remaining_pages)} pages")
        if remaining_pages:
            print(f"Page range: {min(remaining_pages)} to {max(remaining_pages)}")
    else:
        remaining_pages = []
        print("No pages found to process")

    return processed_pages, remaining_pages, all_page_numbers

In [ ]:
processed, remaining, all_pages = get_remaining_pages(base_directory)

Already processed: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 194, 195, 196, 198, 199, 200, 202, 206, 207, 209, 210, 211, 214, 218, 220, 226]
Total pages available: 404
Remaining to process: 19

In [ ]:
# Process only the remaining pages
if remaining:
  print(f"\nStarting batch processing from page {remaining[0]}...")
  results = batch_process_pages(
    start_page=min(remaining),
    end_page=max(remaining),
    client=client,
    model_id=MODEL_ID,
    instructions=instructions,
    example_parts=example_parts,
    base_dir=base_directory,
    skip_existing=True
  )
else:
    print("All pages already processed!")

We will check for completeness and then re-run the files that do no have an output

In [ ]:
# Path setup
pdf_path = Path(f"/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/{base_directory}.pdf")
output_dir = Path(f"/content/drive/MyDrive/merve-february2026/{base_directory}/gemini_output")

# Check if PDF exists
if not pdf_path.exists():
    print(f"PDF not found: {pdf_path}")
else:
    # Get number of pages in PDF
    reader = PdfReader(pdf_path)
    pdf_page_count = len(reader.pages)
    print(f"PDF has {pdf_page_count} pages")

    # Count JSON files in output directory
    if not output_dir.exists():
        print(f"Output directory not found: {output_dir}")
        json_file_count = 0
    else:
        json_files = list(output_dir.glob("page_*.json"))
        json_file_count = len(json_files)
        print(f"Found {json_file_count} JSON files in output directory")

    # Compare
    if pdf_page_count == json_file_count:
        print(f"All pages processed! ({pdf_page_count} pages)")
    else:
        missing = pdf_page_count - json_file_count
        print(f"Missing {missing} pages (PDF: {pdf_page_count}, JSON files: {json_file_count})")

PDF has 682 pages
Found 677 JSON files in output directory
Missing 5 pages (PDF: 682, JSON files: 677)


In [ ]:
# Check for files with null raw_response
if not output_dir.exists():
    print(f"Output directory not found: {output_dir}")
else:
    json_files = sorted(output_dir.glob("page_*.json"))

    if not json_files:
        print("No JSON files found")
    else:
        null_responses = []

        for json_file in json_files:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

                # Check if raw_response is null or missing
                if data.get('raw_response') is None:
                    page_num = data.get('page', 'unknown')
                    null_responses.append({
                        'file': json_file.name,
                        'page': page_num
                    })

        # Report results
        total_files = len(json_files)
        if not null_responses:
            print(f"All {total_files} files have text output")
        else:
            print(f"Found {len(null_responses)} files with null raw_response out of {total_files} total:")
            for item in null_responses:
                print(f"  - {item['file']}")

In [ ]:
# this is if you are missing specific pages and want to re-run only in those

def batch_process_specific_pages(page_list, client, model_id, instructions, example_parts, base_dir, output_suffix=None):
    """
    Process specific pages (e.g., failed pages from a previous run).

    Args:
        page_list: List of page numbers to process
        output_suffix: If provided, creates separate output folder (e.g., "second_pass"). If not, overrides the page
    """
    if output_suffix:
        output_dir = Path(base_dir) / f'gemini_output_{output_suffix}'
    else:
        output_dir = Path(base_dir) / 'gemini_output'

    output_dir.mkdir(parents=True, exist_ok=True)

    jsonl_file = output_dir / 'all_responses.jsonl'
    results_summary = []

    # Retry parameters
    base_delay = 10
    max_delay = 120
    max_retries_rate_limit = 5
    max_retries_other = 3

    for idx, page_num in enumerate(page_list, 1):
        print(f"[{idx}/{len(page_list)}] Processing page {page_num}...")

        # Retry loop
        retry_count = 0
        success = False
        final_data = None

        while retry_count < max_retries_rate_limit:
            success, data = process_page(
                page_num, client, model_id, instructions, example_parts, base_dir
            )

            error_msg = data.get("error", "")

            # Determine if we should retry
            should_retry = False
            retry_type = None

            if not success:
                if "RESOURCE_EXHAUSTED" in error_msg or "429" in error_msg:
                    should_retry = True
                    retry_type = "rate_limit"
                    max_retries = max_retries_rate_limit
                    retry_wait = min(base_delay * (2 ** retry_count), max_delay)
                elif "Empty response" in error_msg or "NO_CANDIDATES" in error_msg:
                    should_retry = True
                    retry_type = "empty_response"
                    max_retries = max_retries_other
                    retry_wait = 3
                elif "not subscriptable" in error_msg or "NoneType" in error_msg:
                    should_retry = True
                    retry_type = "api_error"
                    max_retries = max_retries_other
                    retry_wait = 3
                else:
                    should_retry = retry_count == 0
                    retry_type = "unknown"
                    max_retries = 1
                    retry_wait = 2

            if should_retry and retry_count < max_retries:
                retry_count += 1
                print(f"⚠ {retry_type} (attempt {retry_count}/{max_retries}): {error_msg}")
                print(f"Retrying in {retry_wait} seconds...")
                time.sleep(retry_wait)
                continue
            else:
                final_data = data
                break

        # Save results
        individual_file = output_dir / f'page_{page_num}.json'

        if success:
            with open(jsonl_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

            with open(individual_file, 'w', encoding='utf-8') as f:
                json.dump(final_data, f, ensure_ascii=False, indent=2)

            retry_msg = f" (after {retry_count} retries)" if retry_count > 0 else ""
            print(f"✓ Page {page_num} saved{retry_msg}")
            results_summary.append({
                "page": page_num,
                "status": "success",
                "retries": retry_count
            })

        else:
            print(f"✗ Page {page_num} failed: {final_data.get('error', 'Unknown error')}")

            with open(jsonl_file, 'a', encoding='utf-8') as f:
                f.write(json.dumps(final_data, ensure_ascii=False) + '\n')

            results_summary.append({
                "page": page_num,
                "status": "failed",
                "error": final_data.get('error'),
                "finish_reason": final_data.get('finish_reason'),
                "retries": retry_count
            })

        # Wait before next page
        if idx < len(page_list):
            print(f"Waiting {base_delay} seconds before next page...")
            time.sleep(base_delay)

    # Save summary
    summary_file = output_dir / 'processing_summary.json'
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump({
            "total_pages": len(results_summary),
            "successful": sum(1 for r in results_summary if r["status"] == "success"),
            "failed": sum(1 for r in results_summary if r["status"] == "failed"),
            "results": results_summary
        }, f, indent=2)

    print(f"\n{'='*50}")
    print(f"Processing complete!")
    print(f"Successful: {sum(1 for r in results_summary if r['status'] == 'success')}")
    print(f"Failed: {sum(1 for r in results_summary if r['status'] == 'failed')}")
    print(f"Results saved to: {output_dir}")
    print(f"{'='*50}")

    return results_summary


In [ ]:
from pathlib import Path
from pypdf import PdfReader

# Path setup
pdf_path = Path(f"/content/drive/MyDrive/merve-february2026/just_ottoman_pdfs/{base_directory}.pdf")
transliteration_pages_dir = Path(f"/content/drive/MyDrive/merve-february2026/{base_directory}/transliteration_pages")
output_dir = Path(f"/content/drive/MyDrive/merve-february2026/{base_directory}/gemini_output")

# Check if PDF exists
if not pdf_path.exists():
    print(f"PDF not found: {pdf_path}")
    missing_pages = []
else:
    # Get number of pages in PDF
    reader = PdfReader(pdf_path)
    pdf_page_count = len(reader.pages)
    print(f"PDF has {pdf_page_count} pages")

    # Get the actual page range from transliteration_pages folder
    if not transliteration_pages_dir.exists():
        print(f"Transliteration pages directory not found: {transliteration_pages_dir}")
        missing_pages = []
    else:
        # Get all PDF files in transliteration_pages
        page_pdfs = list(transliteration_pages_dir.glob("page_*.pdf"))

        if not page_pdfs:
            print(f"No page PDFs found in {transliteration_pages_dir}")
            missing_pages = []
        else:
            # Extract page numbers from PDF filenames
            pdf_page_numbers = set()
            for page_pdf in page_pdfs:
                try:
                    page_num = int(page_pdf.stem.split('_')[1])
                    pdf_page_numbers.add(page_num)
                except (IndexError, ValueError):
                    print(f"Warning: Could not parse page number from {page_pdf.name}")

            if not pdf_page_numbers:
                print("Could not extract any page numbers from PDF files")
                missing_pages = []
            else:
                start_page = min(pdf_page_numbers)
                end_page = max(pdf_page_numbers)
                print(f"Page range from transliteration PDFs: {start_page} to {end_page} ({len(pdf_page_numbers)} pages)")

                # Get processed JSON files
                if not output_dir.exists():
                    print(f"Output directory not found: {output_dir}")
                    processed_pages = set()
                else:
                    json_files = list(output_dir.glob("page_*.json"))
                    json_file_count = len(json_files)
                    print(f"Found {json_file_count} JSON files in output directory")

                    # Extract page numbers from JSON filenames
                    processed_pages = set()
                    for json_file in json_files:
                        try:
                            page_num = int(json_file.stem.split('_')[1])
                            processed_pages.add(page_num)
                        except (IndexError, ValueError):
                            print(f"Warning: Could not parse page number from {json_file.name}")

                # Find missing pages (pages that exist in PDF but not in JSON)
                missing_pages = sorted(list(pdf_page_numbers - processed_pages))

                # Compare
                if len(pdf_page_numbers) == len(processed_pages) and not missing_pages:
                    print(f"✓ All pages processed! ({len(pdf_page_numbers)} pages, from {start_page} to {end_page})")
                else:
                    print(f"✗ Missing {len(missing_pages)} pages")
                    print(f"   Expected: {len(pdf_page_numbers)} pages ({start_page}-{end_page})")
                    print(f"   Processed: {len(processed_pages)} JSON files")
                    print(f"   Missing page numbers: {missing_pages}")

# Now you can use the missing_pages list
print(f"\nMissing pages list: {missing_pages}")

In [ ]:
# Step 1: Find pages with no response
pages_to_reprocess = missing_pages
#pages_to_reprocess = [you can also insert an index for any page you want]


# Step 2: Reprocess to a separate folder (recommended)
if pages_to_reprocess:
    print(f"\nReprocessing {len(pages_to_reprocess)} pages to gemini_output_second_pass...")
    results = batch_process_specific_pages(
        page_list=pages_to_reprocess,
        client=client,
        model_id=MODEL_ID,
        instructions=instructions,
        example_parts=example_parts,
        base_dir=base_directory,
        output_suffix="second_pass"  # Creates gemini_output_second_pass
    )
else:
    print("All pages have valid responses!")

# Optional: If you want to overwrite instead, use:
# output_suffix=None